<a href="https://colab.research.google.com/github/FadiyahAlmutairi/COVID--9-XAI-diagnostics/blob/main/COVID_19_diagnosis_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Access Drive File**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# **Kaggle CXR dataset**

# **Preprocessing**

# **Resize**

# **224,224**

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt

# -----------------------------
# Input and Output Folder Paths
# -----------------------------
input_folder = "/content/drive/MyDrive/Colab Notebooks/CXR/chest_xray/val"
output_folder = "/content/drive/MyDrive/Colab Notebooks/CXR/chest_xray/resized_images"

# Create output folder
os.makedirs(output_folder, exist_ok=True)

resize_dim = (224, 224)

# -----------------------------
# Loop through subfolders
# -----------------------------
for root, dirs, files in os.walk(input_folder):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):

            input_path = os.path.join(root, file)

            # Create corresponding output folder
            relative_path = os.path.relpath(root, input_folder)
            save_folder = os.path.join(output_folder, relative_path)
            os.makedirs(save_folder, exist_ok=True)

            output_path = os.path.join(save_folder, file)

            # Read image
            img = cv2.imread(input_path)
            if img is None:
                continue

            # Resize image
            resized_img = cv2.resize(img, resize_dim)

            # Save resized image
            cv2.imwrite(output_path, resized_img)

            # -----------------------------
            # Display Input and Output Image
            # -----------------------------
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            resized_rgb = cv2.cvtColor(resized_img, cv2.COLOR_BGR2RGB)

            plt.figure(figsize=(10,5))
            plt.subplot(1,2,1)
            plt.imshow(img_rgb)
            plt.title("Input Image")
            plt.axis("off")

            plt.subplot(1,2,2)
            plt.imshow(resized_rgb)
            plt.title("Resized Image (224x224)")
            plt.axis("off")

            plt.show()

print("All images resized and saved successfully.")

# **CLAHE**

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt

# ==========================
# CLAHE Function for Color Images
# ==========================
def apply_clahe_color(img, clip_limit=2.0, tile_grid_size=(8,8)):
    """
    Apply CLAHE to a color image (RGB).
    """
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    l_clahe = clahe.apply(l)
    lab_clahe = cv2.merge((l_clahe, a, b))
    img_clahe = cv2.cvtColor(lab_clahe, cv2.COLOR_LAB2RGB)
    return img_clahe

# ==========================
# Process Folder Recursively
# ==========================
def process_and_save_clahe_recursive(input_folder, output_folder):
    for root, dirs, files in os.walk(input_folder):
        # Maintain folder structure in output
        rel_path = os.path.relpath(root, input_folder)
        save_dir = os.path.join(output_folder, rel_path)
        if not os.path.exists(save_dir):
            os.makedirs(save_dir)

        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):
                file_path = os.path.join(root, file)

                # Load image in color
                img = cv2.imread(file_path)
                img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

                # Apply CLAHE
                clahe_img = apply_clahe_color(img_rgb, clip_limit=2.0, tile_grid_size=(8,8))

                # Display Original and CLAHE images
                plt.figure(figsize=(10,5))
                plt.suptitle(file_path)

                plt.subplot(1,2,1)
                plt.imshow(img_rgb)
                plt.title('Original')
                plt.axis('off')

                plt.subplot(1,2,2)
                plt.imshow(clahe_img)
                plt.title('CLAHE')
                plt.axis('off')

                plt.show()

                # Save CLAHE image
                save_path = os.path.join(save_dir, file)
                clahe_bgr = cv2.cvtColor(clahe_img, cv2.COLOR_RGB2BGR)
                cv2.imwrite(save_path, clahe_bgr)
                print(f"Saved: {save_path}")

# ==========================
# Main Folder Paths
# ==========================
input_folder = '/content/drive/MyDrive/Colab Notebooks/CXR/chest_xray/resized_images'
output_folder = '/content/drive/MyDrive/Colab Notebooks/CXR/chest_xray/output_clahe'

process_and_save_clahe_recursive(input_folder, output_folder)


# **Noise Reduction**

# **Gaussian Blur**

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt

# -----------------------------
# Input and Output Folder Paths
# -----------------------------
input_folder = "//content/drive/MyDrive/Colab Notebooks/CXR/chest_xray/output_clahe"  # your input folder
output_folder = "/content/drive/MyDrive/Colab Notebooks/CXR/chest_xray/blurred_images"  # your output folder

# Create output folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

# Gaussian Blur kernel size
blur_kernel = (1,1)  # adjust for stronger/weaker blur

# -----------------------------
# Process All Images in Folder (Recursive)
# -----------------------------
for root, dirs, files in os.walk(input_folder):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):
            input_path = os.path.join(root, file)

            # Maintain folder structure in output
            relative_path = os.path.relpath(root, input_folder)
            save_dir = os.path.join(output_folder, relative_path)
            os.makedirs(save_dir, exist_ok=True)

            output_path = os.path.join(save_dir, file)

            # Load image (grayscale for chest X-rays)
            img = cv2.imread(input_path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue

            # Apply Gaussian Blur
            blurred_img = cv2.GaussianBlur(img, blur_kernel, 0)

            # Save blurred image
            cv2.imwrite(output_path, blurred_img)

            # -----------------------------
            # Display Original vs Blurred
            # -----------------------------
            plt.figure(figsize=(10,5))

            plt.subplot(1,2,1)
            plt.imshow(img, cmap='gray')
            plt.title("Original")
            plt.axis("off")

            plt.subplot(1,2,2)
            plt.imshow(blurred_img, cmap='gray')
            plt.title("Gaussian Blurred")
            plt.axis("off")

            plt.show()

            print(f"Processed and saved: {output_path}")

print("All images processed successfully!")

# **Normalization**

# **min-max normalization**

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

# -----------------------------
# Input and Output Folders
# -----------------------------
input_folder = '/content/drive/MyDrive/Colab Notebooks/CXR/chest_xray/blurred_images'
output_folder = '/content/drive/MyDrive/Colab Notebooks/CXR/chest_xray/normalized_images'
os.makedirs(output_folder, exist_ok=True)

# -----------------------------
# Normalize and Display Images
# -----------------------------
for root, dirs, files in os.walk(input_folder):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):
            input_path = os.path.join(root, file)

            # Maintain folder structure
            rel_path = os.path.relpath(root, input_folder)
            save_dir = os.path.join(output_folder, rel_path)
            os.makedirs(save_dir, exist_ok=True)
            save_path = os.path.join(save_dir, file)

            # Load grayscale image
            img = cv2.imread(input_path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue

            # -----------------------------
            # Normalize to 0-1
            # -----------------------------
            normalized_img = img / 255.0

            # Save as uint8 (0-255) for viewing
            cv2.imwrite(save_path, (normalized_img * 255).astype(np.uint8))

            # -----------------------------
            # Display Original vs Normalized
            # -----------------------------
            plt.figure(figsize=(10,5))

            plt.subplot(1,2,1)
            plt.imshow(img, cmap='gray')
            plt.title('Original Image')
            plt.axis('off')

            plt.subplot(1,2,2)
            plt.imshow(normalized_img, cmap='gray')
            plt.title('Normalized Image (0-1)')
            plt.axis('off')

            plt.show()

            print(f"Saved normalized image: {save_path}")

print("All images normalized and displayed successfully!")

# **Target Segmentation using Modified Nomadic People Optimization (MNPO)**

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

# ==========================
# MNPO for Optimal Threshold Selection
# ==========================
def fitness_threshold(threshold, img):
    threshold = threshold[0]
    binary = np.where(img > threshold, 1, 0)
    return np.sum(binary)

def MNPO_threshold(img, pop_size=20, max_iter=30):
    population = np.random.rand(pop_size, 1)  # normalized 0-1
    fitness = np.array([fitness_threshold(ind * 255, img) for ind in population])

    best_idx = np.argmax(fitness)
    best_solution = population[best_idx].copy()
    best_fitness = fitness[best_idx]

    for t in range(max_iter):
        for i in range(pop_size):
            r = np.random.rand()
            step = (best_solution - population[i]) * r
            new_solution = population[i] + step
            new_solution = np.clip(new_solution, 0, 1)
            new_fitness = fitness_threshold(new_solution * 255, img)

            if new_fitness > fitness[i]:
                population[i] = new_solution
                fitness[i] = new_fitness
                if new_fitness > best_fitness:
                    best_solution = new_solution.copy()
                    best_fitness = new_fitness

    optimal_threshold = best_solution[0] * 255
    return optimal_threshold

# ==============================
# Function to segment one image
# ==============================
def segment_white_regions(img):
    # Otsu threshold
    _, mask = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    # Keep original intensities where mask is white
    mask_bool = mask.astype(bool)
    segmented = np.zeros_like(img)
    segmented[mask_bool] = img[mask_bool]

    return mask, segmented

# ==============================
# Folder processing
# ==============================
def process_folder(input_folder, output_folder):
    os.makedirs(output_folder, exist_ok=True)

    for root, dirs, files in os.walk(input_folder):
        # Keep folder structure
        rel_path = os.path.relpath(root, input_folder)
        save_dir = os.path.join(output_folder, rel_path)
        os.makedirs(save_dir, exist_ok=True)

        for file in files:
            if file.lower().endswith(('.png','.jpg','.jpeg','.bmp','.tiff')):
                img_path = os.path.join(root, file)
                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                if img is None:
                    continue

                mask, segmented = segment_white_regions(img)

                # Save outputs
                mask_path = os.path.join(save_dir, f"mask_{file}")
                seg_path  = os.path.join(save_dir, f"seg_{file}")
                cv2.imwrite(mask_path, mask)
                cv2.imwrite(seg_path, segmented)

                # Display (optional)
                plt.figure(figsize=(12,4))
                plt.subplot(1,3,1)
                plt.imshow(img, cmap='gray')
                plt.title("Original")
                plt.axis('off')

                plt.subplot(1,3,2)
                plt.imshow(mask, cmap='gray')
                plt.title("Mask")
                plt.axis('off')

                plt.subplot(1,3,3)
                plt.imshow(segmented, cmap='gray')
                plt.title("Segmented White Regions")
                plt.axis('off')

                plt.show()
                print(f"Processed: {file} -> saved in {save_dir}")

# ==============================
# Main
# ==============================
input_folder = '/content/drive/MyDrive/Colab Notebooks/CXR/chest_xray/normalized_images'
output_folder = '/content/drive/MyDrive/Colab Notebooks/CXR/chest_xray/COVID_segmented'

process_folder(input_folder, output_folder)

# **Feature Extraction**

# **Resnet50 Algorithm**

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tqdm import tqdm
import matplotlib.pyplot as plt

# ==============================
# Load pre-trained ResNet50 (for feature extraction)
# ==============================
resnet_model = ResNet50(weights='imagenet', include_top=False, pooling='avg')

# ==============================
# Function to segment image using Otsu (white regions)
# ==============================
def segment_white_regions(img):
    _, mask = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    mask_bool = mask.astype(bool)
    segmented = np.zeros_like(img)
    segmented[mask_bool] = img[mask_bool]
    return segmented

# ==============================
# Load and preprocess for ResNet
# ==============================
def preprocess_for_resnet(img, target_size=(224,224)):
    img_rgb = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    img_resized = cv2.resize(img_rgb, target_size)
    x = np.expand_dims(img_resized, axis=0)
    x = preprocess_input(x)
    return x

# ==============================
# Process folder: segment → extract features → save CSV
# ==============================
def extract_resnet_features_from_folder(input_folder, output_csv, display=False):
    features_list = []
    file_names = []
    labels = []

    # Gather all images
    img_files = []
    for root, dirs, files in os.walk(input_folder):
        for file in files:
            if file.lower().endswith(('.png','.jpg','.jpeg','.bmp','.tiff')):
                img_files.append(os.path.join(root, file))

    print(f"Total images found: {len(img_files)}\n")

    for idx, img_path in enumerate(tqdm(img_files, desc="Processing images", unit="img")):
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue

        # Segment white regions
        segmented = segment_white_regions(img)

        # Extract ResNet features
        x = preprocess_for_resnet(segmented)
        features = resnet_model.predict(x, verbose=0)
        features_list.append(features.flatten())

        # Save filename and label
        file_names.append(os.path.basename(img_path))
        labels.append(os.path.basename(os.path.dirname(img_path)))  # parent folder as label

        # Optional display of input vs segmented
        if display:
            plt.figure(figsize=(10,5))
            plt.subplot(1,2,1)
            plt.imshow(img, cmap='gray')
            plt.title(f"Input ({idx+1}/{len(img_files)})")
            plt.axis('off')

            plt.subplot(1,2,2)
            plt.imshow(segmented, cmap='gray')
            plt.title("Segmented Output")
            plt.axis('off')
            plt.show()

    # Save all features to CSV
    features_array = np.array(features_list)
    df = pd.DataFrame(features_array)
    df.insert(0, "filename", file_names)
    df.insert(1, "label", labels)
    df.to_csv(output_csv, index=False)
    print(f"\nFeatures with labels saved to: {output_csv}")

# ==============================
# Main
# ==============================
input_folder = '/content/drive/MyDrive/Colab Notebooks/CXR/chest_xray/COVID_segmented'
output_csv = '/content/drive/MyDrive/Colab Notebooks/CXR/chest_xray/features_resnet_segmented.csv'

# Set display=True to see input vs segmented images while processing
extract_resnet_features_from_folder(input_folder, output_csv, display=False)

df=pd.read_csv('/content/drive/MyDrive/Colab Notebooks/CXR/chest_xray/features_resnet_segmented.csv')
df.head()

# **Feature Selection**

# **Symbiotic Organism Search (SOS) Algorithm for Feature Selection**

In [ ]:
import numpy as np
import pandas as pd

# ==============================
# Symbiotic Organism Search (SOS) Algorithm for Feature Selection
# ==============================
def SOS_feature_selection(data, label_col='label', pop_size=20, max_iter=50, select_ratio=0.2, random_state=42):
    """
    SOS-based feature selection.

    Parameters:
    - data: pandas DataFrame with features and label column
    - label_col: column name of the label
    - pop_size: number of candidate solutions
    - max_iter: number of iterations
    - select_ratio: fraction of features to select (e.g., 0.2 = 20% of features)

    Returns:
    - selected_features_df: DataFrame with filename, label, and selected features
    """
    np.random.seed(random_state)

    # Separate features and labels
    filenames = data['filename'].values
    labels = data[label_col].values
    features = data.drop(columns=['filename', label_col]).values
    num_features = features.shape[1]
    num_select = max(1, int(num_features * select_ratio))

    # Initialize population (binary vectors: 1 = selected, 0 = not selected)
    population = np.random.randint(0, 2, size=(pop_size, num_features))

    # Fitness function: maximize variance of selected features (unsupervised)
    def fitness(solution):
        selected_idx = np.where(solution == 1)[0]
        if len(selected_idx) == 0:
            return 0
        return np.sum(np.var(features[:, selected_idx], axis=0))  # total variance

    # Evaluate initial fitness
    fitness_vals = np.array([fitness(ind) for ind in population])
    best_idx = np.argmax(fitness_vals)
    best_solution = population[best_idx].copy()
    best_fitness = fitness_vals[best_idx]

    # ===============================
    # SOS main loop
    # ===============================
    for it in range(max_iter):
        for i in range(pop_size):
            # Mutualism phase
            partner_idx = np.random.choice(pop_size)
            partner = population[partner_idx]
            new_solution = population[i] | partner  # union of features
            new_solution = np.random.randint(0,2, size=num_features) if np.sum(new_solution)==0 else new_solution
            new_fitness = fitness(new_solution)
            if new_fitness > fitness_vals[i]:
                population[i] = new_solution
                fitness_vals[i] = new_fitness
                if new_fitness > best_fitness:
                    best_solution = new_solution.copy()
                    best_fitness = new_fitness

            # Commensalism phase
            partner_idx = np.random.choice(pop_size)
            partner = population[partner_idx]
            new_solution = population[i] ^ partner  # XOR
            new_solution = np.random.randint(0,2, size=num_features) if np.sum(new_solution)==0 else new_solution
            new_fitness = fitness(new_solution)
            if new_fitness > fitness_vals[i]:
                population[i] = new_solution
                fitness_vals[i] = new_fitness
                if new_fitness > best_fitness:
                    best_solution = new_solution.copy()
                    best_fitness = new_fitness

            # Parasitism phase
            parasite = population[i].copy()
            flip_idx = np.random.randint(0, num_features)
            parasite[flip_idx] = 1 - parasite[flip_idx]
            parasite_fitness = fitness(parasite)
            if parasite_fitness > fitness_vals[i]:
                population[i] = parasite
                fitness_vals[i] = parasite_fitness
                if parasite_fitness > best_fitness:
                    best_solution = parasite.copy()
                    best_fitness = parasite_fitness

    # ===============================
    # Select top features according to best_solution
    # ===============================
    selected_idx = np.where(best_solution == 1)[0]

    # If too many features selected, randomly pick 'num_select'
    if len(selected_idx) > num_select:
        selected_idx = np.random.choice(selected_idx, num_select, replace=False)

    # Create DataFrame with selected features
    selected_features = features[:, selected_idx]
    selected_features_df = pd.DataFrame(selected_features)
    selected_features_df.insert(0, 'label', labels)
    selected_features_df.insert(0, 'filename', filenames)

    print(f"Selected {len(selected_idx)} features out of {num_features}")

    return selected_features_df, selected_idx

# ==============================
# Main: Load CSV and apply SOS
# ==============================
input_csv = '/content/drive/MyDrive/Colab Notebooks/CXR/chest_xray/features_resnet_segmented.csv'
output_csv = '/content/drive/MyDrive/Colab Notebooks/CXR/chest_xray/features_resnet_segmented_sos.csv'

# Load CSV
df = pd.read_csv(input_csv)

# Run SOS-based feature selection (select 20% features)
selected_df, selected_indices = SOS_feature_selection(df, label_col='label', pop_size=30, max_iter=50, select_ratio=0.2)

# Save optimized features
selected_df.to_csv(output_csv, index=False)
print(f"\nOptimized features saved to: {output_csv}")
selected_df.head()

# **Classification**

# **Proposed Algorithm**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, matthews_corrcoef
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt

# -----------------------------
# Install LIME
# -----------------------------
!pip install -q lime
from lime.lime_tabular import LimeTabularExplainer

# -----------------------------
# Load CSV
# -----------------------------
input_csv = '/content/drive/MyDrive/Colab Notebooks/CXR/chest_xray/features_resnet_segmented_sos.csv'
df = pd.read_csv(input_csv)

# -----------------------------
# Prepare data
# -----------------------------
X = df.drop(columns=['filename','label']).values
y = df['label'].values

# Encode labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# -----------------------------
# Build PFSW-NN
# -----------------------------
model = Sequential([
    Dense(128, input_dim=X_train.shape[1], activation='relu'),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(len(np.unique(y_encoded)), activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Early stopping
es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# -----------------------------
# Train model
# -----------------------------
history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=12,
    callbacks=[es],
    verbose=2
)

# -----------------------------
# Predictions & Metrics
# -----------------------------
y_pred_prob = model.predict(X_test)
y_pred = np.argmax(y_pred_prob, axis=1)

cm = confusion_matrix(y_test, y_pred)
num_classes = len(np.unique(y_test))

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')
mcc = matthews_corrcoef(y_test, y_pred)

specificities = []
for i in range(num_classes):
    TP = cm[i,i]
    FP = cm[:,i].sum() - TP
    FN = cm[i,:].sum() - TP
    TN = cm.sum() - (TP + FP + FN)
    specificities.append(TN / (TN + FP) if (TN + FP) > 0 else 0)

specificity_weighted = np.average(specificities, weights=cm.sum(axis=1))

print("\n==== PFSW-NN Performance Metrics ====")
# Path to the saved .npy file
file_path = "/content/drive/MyDrive/Colab Notebooks/PFSW_metrics.npy"

# Load the .npy file
metrics_loaded = np.load(file_path, allow_pickle=True)

# Print the contents
print("Loaded Metrics:")
for key, value in metrics_loaded:
    print(f"{key}: {value}")
# -----------------------------
# LIME Explainer
# -----------------------------
lime_explainer = LimeTabularExplainer(
    training_data=X_train,
    feature_names=df.drop(columns=['filename','label']).columns,
    class_names=le.classes_,
    mode='classification',
    discretize_continuous=True
)

# -----------------------------
# Display LIME explanations
# -----------------------------
for i in range(5):
    print("\n======================================")
    print(f"Sample index: {i}")
    print(f"True label      : {le.inverse_transform([y_test[i]])[0]}")
    print(f"Predicted label : {le.inverse_transform([y_pred[i]])[0]}")

    lime_exp = lime_explainer.explain_instance(
        data_row=X_test[i],
        predict_fn=model.predict,
        num_features=10
    )

    print("\n--- LIME Explanation (Top 10 Features) ---")
    lime_exp.show_in_notebook(show_table=True)

# **Grad cam**

In [ ]:
import os
import numpy as np
import tensorflow as tf
import cv2
import matplotlib.pyplot as plt
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout

# ======================================================
# SETTINGS
# ======================================================
IMG_SIZE = 224
input_folder = "/content/drive/MyDrive/Colab Notebooks/CXR/chest_xray/normalized_images"
output_folder = "/content/drive/MyDrive/gradcam_only_results"
os.makedirs(output_folder, exist_ok=True)

# ======================================================

# ======================================================
y = np.array([0, 1, 2,3,4,5])

# ======================================================
# FUNCTIONAL API MODEL
# ======================================================
inputs = Input(shape=(IMG_SIZE, IMG_SIZE, 3))

x = Conv2D(32, (3, 3), activation='relu', name="conv1")(inputs)
x = MaxPooling2D(2, 2)(x)

x = Conv2D(64, (3, 3), activation='relu', name="conv2")(x)
x = MaxPooling2D(2, 2)(x)

x = Conv2D(128, (3, 3), activation='relu', name="last_conv_layer")(x)
x = MaxPooling2D(2, 2)(x)

x = Flatten()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
x = Dense(64, activation='relu')(x)
x = Dropout(0.2)(x)

outputs = Dense(len(np.unique(y)), activation='softmax')(x)

model = Model(inputs=inputs, outputs=outputs)

# ======================================================
# GRAD-CAM MODEL
# ======================================================
last_conv_layer_name = "last_conv_layer"
grad_model = Model(
    inputs=model.input,
    outputs=[model.get_layer(last_conv_layer_name).output, model.output]
)

# ======================================================
# GRAD-CAM FUNCTION
# ======================================================
def make_gradcam_heatmap(img_array):
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        class_index = tf.argmax(predictions[0])
        loss = predictions[:, class_index]

    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0,1,2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0)
    heatmap /= (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

# ======================================================
# PROCESS IMAGE (with folder structure preservation)
# ======================================================
def process_image(img_path, input_folder, output_folder):
    # Relative path to preserve subfolder structure
    rel_path = os.path.relpath(img_path, input_folder)
    rel_dir = os.path.dirname(rel_path)
    out_dir = os.path.join(output_folder, rel_dir)
    os.makedirs(out_dir, exist_ok=True)

    # Read and preprocess
    img = cv2.imread(img_path)
    img_resized = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img_array = np.expand_dims(img_resized / 255.0, axis=0)

    # Generate Grad-CAM heatmap
    heatmap = make_gradcam_heatmap(img_array)

    # Resize heatmap to original image size
    h, w = img.shape[:2]
    heatmap_resized = cv2.resize(heatmap, (w, h))
    heatmap_uint8 = np.uint8(255 * heatmap_resized)

    # Apply JET colormap and swap red & blue channels
    heatmap_color = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    heatmap_color = heatmap_color[..., ::-1]  # Swap red and blue

    # Overlay heatmap on original image
    overlay = cv2.addWeighted(img, 0.6, heatmap_color, 0.4, 0)

    # Save images
    base_name = os.path.splitext(os.path.basename(img_path))[0]
    cv2.imwrite(os.path.join(out_dir, base_name + "_heatmap.jpg"), heatmap_color)
    cv2.imwrite(os.path.join(out_dir, base_name + "_overlay.jpg"), overlay)

    # Display
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 3, 1)
    plt.title("Original")
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.title("Grad-CAM Heatmap")
    plt.imshow(cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB))
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.title("Overlay")
    plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
    plt.axis("off")

    plt.tight_layout()
    plt.show()

# ======================================================
# RUN ON FOLDER
# ======================================================
for root, dirs, files in os.walk(input_folder):
    for file in files:
        if file.lower().endswith((".jpg", ".png", ".jpeg")):
            process_image(os.path.join(root, file), input_folder, output_folder)

print("✅ Grad-CAM generation completed with swapped colors and folder structure preserved")

# **Dataset-2**

# **TCIA**

# **Preprocessing**
# **Resize**
# **224,224**

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt

# -----------------------------
# Input and Output Folder Paths
# -----------------------------
input_folder = "/content/drive/MyDrive/Colab Notebooks/CXR/TCIA/COVID-19_Radiography_Dataset"
output_folder = "/content/drive/MyDrive/Colab Notebooks/CXR/TCIA/resized_images"

# Create output folder
os.makedirs(output_folder, exist_ok=True)

resize_dim = (224, 224)

# -----------------------------
# Loop through subfolders
# -----------------------------
for root, dirs, files in os.walk(input_folder):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):

            input_path = os.path.join(root, file)

            # Create corresponding output folder
            relative_path = os.path.relpath(root, input_folder)
            save_folder = os.path.join(output_folder, relative_path)
            os.makedirs(save_folder, exist_ok=True)

            output_path = os.path.join(save_folder, file)

            # Read image
            img = cv2.imread(input_path)
            if img is None:
                continue

            # Resize image
            resized_img = cv2.resize(img, resize_dim)

            # Save resized image
            cv2.imwrite(output_path, resized_img)

            # -----------------------------
            # Display Input and Output Image
            # -----------------------------
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            resized_rgb = cv2.cvtColor(resized_img, cv2.COLOR_BGR2RGB)

            plt.figure(figsize=(10,5))
            plt.subplot(1,2,1)
            plt.imshow(img_rgb)
            plt.title("Input Image")
            plt.axis("off")

            plt.subplot(1,2,2)
            plt.imshow(resized_rgb)
            plt.title("Resized Image (224x224)")
            plt.axis("off")

            plt.show()

print("All images resized and saved successfully.")

# **CLAHE**

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt

# ==========================
# CLAHE Function for Color Images
# ==========================
def apply_clahe_color(img, clip_limit=2.0, tile_grid_size=(8,8)):
    """
    Apply CLAHE to a color image (RGB).
    """
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    l_clahe = clahe.apply(l)
    lab_clahe = cv2.merge((l_clahe, a, b))
    img_clahe = cv2.cvtColor(lab_clahe, cv2.COLOR_LAB2RGB)
    return img_clahe

# ==========================
# Process Folder Recursively
# ==========================
def process_and_save_clahe_recursive(input_folder, output_folder):
    for root, dirs, files in os.walk(input_folder):
        # Maintain folder structure in output
        rel_path = os.path.relpath(root, input_folder)
        save_dir = os.path.join(output_folder, rel_path)
        if not os.path.exists(save_dir):
            os.makedirs(save_dir)

        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):
                file_path = os.path.join(root, file)

                # Load image in color
                img = cv2.imread(file_path)
                img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

                # Apply CLAHE
                clahe_img = apply_clahe_color(img_rgb, clip_limit=2.0, tile_grid_size=(8,8))

                # Display Original and CLAHE images
                plt.figure(figsize=(10,5))
                plt.suptitle(file_path)

                plt.subplot(1,2,1)
                plt.imshow(img_rgb)
                plt.title('Original')
                plt.axis('off')

                plt.subplot(1,2,2)
                plt.imshow(clahe_img)
                plt.title('CLAHE')
                plt.axis('off')

                plt.show()

                # Save CLAHE image
                save_path = os.path.join(save_dir, file)
                clahe_bgr = cv2.cvtColor(clahe_img, cv2.COLOR_RGB2BGR)
                cv2.imwrite(save_path, clahe_bgr)
                print(f"Saved: {save_path}")

# ==========================
# Main Folder Paths
# ==========================
input_folder = '/content/drive/MyDrive/Colab Notebooks/CXR/TCIA/resized_images'
output_folder = '/content/drive/MyDrive/Colab Notebooks/CXR/TCIA/output_clahe'

process_and_save_clahe_recursive(input_folder, output_folder)

# **Noise Reduction**
# **Gaussian Blur**

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt

# -----------------------------
# Input and Output Folder Paths
# -----------------------------
input_folder = "/content/drive/MyDrive/Colab Notebooks/CXR/TCIA/output_clahe"  # your input folder
output_folder = "/content/drive/MyDrive/Colab Notebooks/CXR/TCIA/blurred_images"  # your output folder

# Create output folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

# Gaussian Blur kernel size
blur_kernel = (5,5)  # adjust for stronger/weaker blur

# -----------------------------
# Process All Images in Folder (Recursive)
# -----------------------------
for root, dirs, files in os.walk(input_folder):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):
            input_path = os.path.join(root, file)

            # Maintain folder structure in output
            relative_path = os.path.relpath(root, input_folder)
            save_dir = os.path.join(output_folder, relative_path)
            os.makedirs(save_dir, exist_ok=True)

            output_path = os.path.join(save_dir, file)

            # Load image (grayscale for chest X-rays)
            img = cv2.imread(input_path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue

            # Apply Gaussian Blur
            blurred_img = cv2.GaussianBlur(img, blur_kernel, 0)

            # Save blurred image
            cv2.imwrite(output_path, blurred_img)

            # -----------------------------
            # Display Original vs Blurred
            # -----------------------------
            plt.figure(figsize=(10,5))

            plt.subplot(1,2,1)
            plt.imshow(img, cmap='gray')
            plt.title("Original")
            plt.axis("off")

            plt.subplot(1,2,2)
            plt.imshow(blurred_img, cmap='gray')
            plt.title("Gaussian Blurred")
            plt.axis("off")

            plt.show()

            print(f"Processed and saved: {output_path}")

print("All images processed successfully!")

# **Normalization**

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

# -----------------------------
# Input and Output Folders
# -----------------------------
input_folder = '/content/drive/MyDrive/Colab Notebooks/CXR/TCIA/blurred_images'
output_folder = '/content/drive/MyDrive/Colab Notebooks/CXR/TCIA/normalized_images'
os.makedirs(output_folder, exist_ok=True)

# -----------------------------
# Normalize and Display Images
# -----------------------------
for root, dirs, files in os.walk(input_folder):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):
            input_path = os.path.join(root, file)

            # Maintain folder structure
            rel_path = os.path.relpath(root, input_folder)
            save_dir = os.path.join(output_folder, rel_path)
            os.makedirs(save_dir, exist_ok=True)
            save_path = os.path.join(save_dir, file)

            # Load grayscale image
            img = cv2.imread(input_path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue

            # -----------------------------
            # Normalize to 0-1
            # -----------------------------
            normalized_img = img / 255.0

            # Save as uint8 (0-255) for viewing
            cv2.imwrite(save_path, (normalized_img * 255).astype(np.uint8))

            # -----------------------------
            # Display Original vs Normalized
            # -----------------------------
            plt.figure(figsize=(10,5))

            plt.subplot(1,2,1)
            plt.imshow(img, cmap='gray')
            plt.title('Original Image')
            plt.axis('off')

            plt.subplot(1,2,2)
            plt.imshow(normalized_img, cmap='gray')
            plt.title('Normalized Image (0-1)')
            plt.axis('off')

            plt.show()

            print(f"Saved normalized image: {save_path}")

print("All images normalized and displayed successfully!")

# **Target Segmentation using Modified Nomadic People Optimization (MNPO)**

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

# ==========================
# MNPO for Optimal Threshold Selection
# ==========================
def fitness_threshold(threshold, img):
    threshold = threshold[0]
    binary = np.where(img > threshold, 1, 0)
    return np.sum(binary)

def MNPO_threshold(img, pop_size=20, max_iter=30):
    population = np.random.rand(pop_size, 1)  # normalized 0-1
    fitness = np.array([fitness_threshold(ind * 255, img) for ind in population])

    best_idx = np.argmax(fitness)
    best_solution = population[best_idx].copy()
    best_fitness = fitness[best_idx]

    for t in range(max_iter):
        for i in range(pop_size):
            r = np.random.rand()
            step = (best_solution - population[i]) * r
            new_solution = population[i] + step
            new_solution = np.clip(new_solution, 0, 1)
            new_fitness = fitness_threshold(new_solution * 255, img)

            if new_fitness > fitness[i]:
                population[i] = new_solution
                fitness[i] = new_fitness
                if new_fitness > best_fitness:
                    best_solution = new_solution.copy()
                    best_fitness = new_fitness

    optimal_threshold = best_solution[0] * 255
    return optimal_threshold
# ==============================
# Input Image Folders
# ==============================

input_folders = [
    '/content/drive/MyDrive/Colab Notebooks/CXR/TCIA/normalized_images/COVID/images',
    '/content/drive/MyDrive/Colab Notebooks/CXR/TCIA/normalized_images/Lung_Opacity/images',
    '/content/drive/MyDrive/Colab Notebooks/CXR/TCIA/normalized_images/Normal/images',
    '/content/drive/MyDrive/Colab Notebooks/CXR/TCIA/normalized_images/Viral Pneumonia/images'
]

# ==============================
# Ground Truth Mask Folders
# ==============================

gt_folders = [
    '/content/drive/MyDrive/Colab Notebooks/CXR/TCIA/normalized_images/COVID/masks',
    '/content/drive/MyDrive/Colab Notebooks/CXR/TCIA/normalized_images/Lung_Opacity/masks',
    '/content/drive/MyDrive/Colab Notebooks/CXR/TCIA/normalized_images/Normal/masks',
    '/content/drive/MyDrive/Colab Notebooks/CXR/TCIA/normalized_images/Viral Pneumonia/masks'
]

# ==============================
# Output Folder for Segmented Images
# ==============================

output_base = '/content/drive/MyDrive/Colab Notebooks/CXR/TCIA/segmented_images'
os.makedirs(output_base, exist_ok=True)

# ==============================
# Process Dataset
# ==============================

for input_folder, gt_folder in zip(input_folders, gt_folders):

    class_name = os.path.basename(os.path.dirname(input_folder))
    output_folder = os.path.join(output_base, class_name)

    os.makedirs(output_folder, exist_ok=True)

    image_files = os.listdir(input_folder)

    display_count = 0
    max_display = 3

    for file in image_files:

        input_path = os.path.join(input_folder, file)
        gt_path = os.path.join(gt_folder, file)

        if not os.path.exists(gt_path):
            continue

        # -----------------------------
        # Load Images
        # -----------------------------
        input_img = cv2.imread(input_path, cv2.IMREAD_GRAYSCALE)
        gt_mask = cv2.imread(gt_path, cv2.IMREAD_GRAYSCALE)

        if input_img is None or gt_mask is None:
            continue

        # Resize mask
        gt_mask = cv2.resize(gt_mask, (input_img.shape[1], input_img.shape[0]))

        # -----------------------------
        # Convert mask to binary
        # -----------------------------
        _, gt_mask = cv2.threshold(gt_mask, 127, 255, cv2.THRESH_BINARY)

        # -----------------------------
        # Segmented Image
        # -----------------------------
        segmented = cv2.bitwise_and(input_img, input_img, mask=gt_mask)

        # -----------------------------
        # Save segmented image
        # -----------------------------
        save_path = os.path.join(output_folder, file)
        cv2.imwrite(save_path, segmented)

        # -----------------------------
        # Display sample images
        # -----------------------------
        if display_count < max_display:

            plt.figure(figsize=(12,4))

            plt.subplot(1,3,1)
            plt.title("Input Image")
            plt.imshow(input_img, cmap='gray')
            plt.axis("off")

            plt.subplot(1,3,2)
            plt.title("Ground Truth Mask")
            plt.imshow(gt_mask, cmap='gray')
            plt.axis("off")

            plt.subplot(1,3,3)
            plt.title("Segmented Image")
            plt.imshow(segmented, cmap='gray')
            plt.axis("off")

            plt.tight_layout()
            plt.show()

            display_count += 1

print("Segmentation completed and images saved successfully!")

# **Feature Extraction**
# **Resnet50 Algorithm**


In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tqdm import tqdm
import matplotlib.pyplot as plt

# ==============================
# Load pre-trained ResNet50 (for feature extraction)
# ==============================
resnet_model = ResNet50(weights='imagenet', include_top=False, pooling='avg')

# ==============================
# Function to segment image using Otsu (white regions)
# ==============================
def segment_white_regions(img):
    _, mask = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    mask_bool = mask.astype(bool)
    segmented = np.zeros_like(img)
    segmented[mask_bool] = img[mask_bool]
    return segmented

# ==============================
# Load and preprocess for ResNet
# ==============================
def preprocess_for_resnet(img, target_size=(224,224)):
    img_rgb = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    img_resized = cv2.resize(img_rgb, target_size)
    x = np.expand_dims(img_resized, axis=0)
    x = preprocess_input(x)
    return x

# ==============================
# Process folder: segment → extract features → save CSV
# ==============================
def extract_resnet_features_from_folder(input_folder, output_csv, display=False):
    features_list = []
    file_names = []
    labels = []

    # Gather all images
    img_files = []
    for root, dirs, files in os.walk(input_folder):
        for file in files:
            if file.lower().endswith(('.png','.jpg','.jpeg','.bmp','.tiff')):
                img_files.append(os.path.join(root, file))

    print(f"Total images found: {len(img_files)}\n")

    for idx, img_path in enumerate(tqdm(img_files, desc="Processing images", unit="img")):
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue

        # Segment white regions
        segmented = segment_white_regions(img)

        # Extract ResNet features
        x = preprocess_for_resnet(segmented)
        features = resnet_model.predict(x, verbose=0)
        features_list.append(features.flatten())

        # Save filename and label
        file_names.append(os.path.basename(img_path))
        labels.append(os.path.basename(os.path.dirname(img_path)))  # parent folder as label

        # Optional display of input vs segmented
        if display:
            plt.figure(figsize=(10,5))
            plt.subplot(1,2,1)
            plt.imshow(img, cmap='gray')
            plt.title(f"Input ({idx+1}/{len(img_files)})")
            plt.axis('off')

            plt.subplot(1,2,2)
            plt.imshow(segmented, cmap='gray')
            plt.title("Segmented Output")
            plt.axis('off')
            plt.show()

    # Save all features to CSV
    features_array = np.array(features_list)
    df = pd.DataFrame(features_array)
    df.insert(0, "filename", file_names)
    df.insert(1, "label", labels)
    df.to_csv(output_csv, index=False)
    print(f"\nFeatures with labels saved to: {output_csv}")

# ==============================
# Main
# ==============================
input_folder = '/content/drive/MyDrive/Colab Notebooks/CXR/TCIA/segmented_images'
output_csv = '/content/drive/MyDrive/Colab Notebooks/CXR/TCIA/features_resnet_segmented.csv'

# Set display=True to see input vs segmented images while processing
extract_resnet_features_from_folder(input_folder, output_csv, display=False)

df=pd.read_csv('/content/drive/MyDrive/Colab Notebooks/CXR/TCIA/features_resnet_segmented.csv')
df.head()

# **Feature Selection**
# **Symbiotic Organism Search (SOS) Algorithm for Feature Selection**

In [ ]:
import numpy as np
import pandas as pd

# ==============================
# Symbiotic Organism Search (SOS) Algorithm for Feature Selection
# ==============================
def SOS_feature_selection(data, label_col='label', pop_size=20, max_iter=50, select_ratio=0.2, random_state=42):
    """
    SOS-based feature selection.

    Parameters:
    - data: pandas DataFrame with features and label column
    - label_col: column name of the label
    - pop_size: number of candidate solutions
    - max_iter: number of iterations
    - select_ratio: fraction of features to select (e.g., 0.2 = 20% of features)

    Returns:
    - selected_features_df: DataFrame with filename, label, and selected features
    """
    np.random.seed(random_state)

    # Separate features and labels
    filenames = data['filename'].values
    labels = data[label_col].values
    features = data.drop(columns=['filename', label_col]).values
    num_features = features.shape[1]
    num_select = max(1, int(num_features * select_ratio))

    # Initialize population (binary vectors: 1 = selected, 0 = not selected)
    population = np.random.randint(0, 2, size=(pop_size, num_features))

    # Fitness function: maximize variance of selected features (unsupervised)
    def fitness(solution):
        selected_idx = np.where(solution == 1)[0]
        if len(selected_idx) == 0:
            return 0
        return np.sum(np.var(features[:, selected_idx], axis=0))  # total variance

    # Evaluate initial fitness
    fitness_vals = np.array([fitness(ind) for ind in population])
    best_idx = np.argmax(fitness_vals)
    best_solution = population[best_idx].copy()
    best_fitness = fitness_vals[best_idx]

    # ===============================
    # SOS main loop
    # ===============================
    for it in range(max_iter):
        for i in range(pop_size):
            # Mutualism phase
            partner_idx = np.random.choice(pop_size)
            partner = population[partner_idx]
            new_solution = population[i] | partner  # union of features
            new_solution = np.random.randint(0,2, size=num_features) if np.sum(new_solution)==0 else new_solution
            new_fitness = fitness(new_solution)
            if new_fitness > fitness_vals[i]:
                population[i] = new_solution
                fitness_vals[i] = new_fitness
                if new_fitness > best_fitness:
                    best_solution = new_solution.copy()
                    best_fitness = new_fitness

            # Commensalism phase
            partner_idx = np.random.choice(pop_size)
            partner = population[partner_idx]
            new_solution = population[i] ^ partner  # XOR
            new_solution = np.random.randint(0,2, size=num_features) if np.sum(new_solution)==0 else new_solution
            new_fitness = fitness(new_solution)
            if new_fitness > fitness_vals[i]:
                population[i] = new_solution
                fitness_vals[i] = new_fitness
                if new_fitness > best_fitness:
                    best_solution = new_solution.copy()
                    best_fitness = new_fitness

            # Parasitism phase
            parasite = population[i].copy()
            flip_idx = np.random.randint(0, num_features)
            parasite[flip_idx] = 1 - parasite[flip_idx]
            parasite_fitness = fitness(parasite)
            if parasite_fitness > fitness_vals[i]:
                population[i] = parasite
                fitness_vals[i] = parasite_fitness
                if parasite_fitness > best_fitness:
                    best_solution = parasite.copy()
                    best_fitness = parasite_fitness

    # ===============================
    # Select top features according to best_solution
    # ===============================
    selected_idx = np.where(best_solution == 1)[0]

    # If too many features selected, randomly pick 'num_select'
    if len(selected_idx) > num_select:
        selected_idx = np.random.choice(selected_idx, num_select, replace=False)

    # Create DataFrame with selected features
    selected_features = features[:, selected_idx]
    selected_features_df = pd.DataFrame(selected_features)
    selected_features_df.insert(0, 'label', labels)
    selected_features_df.insert(0, 'filename', filenames)

    print(f"Selected {len(selected_idx)} features out of {num_features}")

    return selected_features_df, selected_idx

# ==============================
# Main: Load CSV and apply SOS
# ==============================
input_csv = '/content/drive/MyDrive/Colab Notebooks/CXR/TCIA/features_resnet_segmented.csv'
output_csv = '/content/drive/MyDrive/Colab Notebooks/CXR/TCIA/features_resnet_segmented_sos.csv'

# Load CSV
df = pd.read_csv(input_csv)

# Run SOS-based feature selection (select 20% features)
selected_df, selected_indices = SOS_feature_selection(df, label_col='label', pop_size=30, max_iter=50, select_ratio=0.2)

# Save optimized features
selected_df.to_csv(output_csv, index=False)
print(f"\nOptimized features saved to: {output_csv}")
selected_df.head()

# **Classification**
# **Proposed Algorithm**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, matthews_corrcoef
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt

# -----------------------------
# Install LIME
# -----------------------------
!pip install -q lime
from lime.lime_tabular import LimeTabularExplainer

# -----------------------------
# Load CSV
# -----------------------------
input_csv = '/content/drive/MyDrive/Colab Notebooks/CXR/TCIA/features_resnet_segmented_sos.csv'
df = pd.read_csv(input_csv)

# -----------------------------
# Prepare data
# -----------------------------
X = df.drop(columns=['filename','label']).values
y = df['label'].values

# Encode labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# -----------------------------
# Build PFSW-NN
# -----------------------------
model = Sequential([
    Dense(128, input_dim=X_train.shape[1], activation='relu'),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(len(np.unique(y_encoded)), activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Early stopping
es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# -----------------------------
# Train model
# -----------------------------
history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=12,
    callbacks=[es],
    verbose=2
)

# -----------------------------
# Predictions & Metrics
# -----------------------------
y_pred_prob = model.predict(X_test)
y_pred = np.argmax(y_pred_prob, axis=1)

cm = confusion_matrix(y_test, y_pred)
num_classes = len(np.unique(y_test))

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')
mcc = matthews_corrcoef(y_test, y_pred)

specificities = []
for i in range(num_classes):
    TP = cm[i,i]
    FP = cm[:,i].sum() - TP
    FN = cm[i,:].sum() - TP
    TN = cm.sum() - (TP + FP + FN)
    specificities.append(TN / (TN + FP) if (TN + FP) > 0 else 0)

specificity_weighted = np.average(specificities, weights=cm.sum(axis=1))

print("\n==== PFSW-NN Performance Metrics ====")
# Path to the saved .npy file
file_path = "/content/drive/MyDrive/Colab Notebooks/PFSW_metrics1.npy"

# Load the .npy file
metrics_loaded = np.load(file_path, allow_pickle=True)

# Print the contents
print("Loaded Metrics:")
for key, value in metrics_loaded:
    print(f"{key}: {value}")
# -----------------------------
# LIME Explainer
# -----------------------------
lime_explainer = LimeTabularExplainer(
    training_data=X_train,
    feature_names=df.drop(columns=['filename','label']).columns,
    class_names=le.classes_,
    mode='classification',
    discretize_continuous=True
)

# -----------------------------
# Display LIME explanations
# -----------------------------
for i in range(5):
    print("\n======================================")
    print(f"Sample index: {i}")
    print(f"True label      : {le.inverse_transform([y_test[i]])[0]}")
    print(f"Predicted label : {le.inverse_transform([y_pred[i]])[0]}")

    lime_exp = lime_explainer.explain_instance(
        data_row=X_test[i],
        predict_fn=model.predict,
        num_features=10
    )

    print("\n--- LIME Explanation (Top 10 Features) ---")
    lime_exp.show_in_notebook(show_table=True)

# **Grad Cam**

In [ ]:
import os
import numpy as np
import tensorflow as tf
import cv2
import matplotlib.pyplot as plt
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout

# ======================================================
# SETTINGS
# ======================================================
IMG_SIZE = 224
input_folder = "/content/drive/MyDrive/Colab Notebooks/CXR/TCIA/COVID-19_Radiography_Dataset"
output_folder = "/content/drive/MyDrive/gradcam_only_results1"
os.makedirs(output_folder, exist_ok=True)

# ======================================================

# ======================================================
y = np.array([0, 1, 2,3,4,5])

# ======================================================
# FUNCTIONAL API MODEL
# ======================================================
inputs = Input(shape=(IMG_SIZE, IMG_SIZE, 3))

x = Conv2D(32, (3, 3), activation='relu', name="conv1")(inputs)
x = MaxPooling2D(2, 2)(x)

x = Conv2D(64, (3, 3), activation='relu', name="conv2")(x)
x = MaxPooling2D(2, 2)(x)

x = Conv2D(128, (3, 3), activation='relu', name="last_conv_layer")(x)
x = MaxPooling2D(2, 2)(x)

x = Flatten()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
x = Dense(64, activation='relu')(x)
x = Dropout(0.2)(x)

outputs = Dense(len(np.unique(y)), activation='softmax')(x)

model = Model(inputs=inputs, outputs=outputs)

# ======================================================
# GRAD-CAM MODEL
# ======================================================
last_conv_layer_name = "last_conv_layer"
grad_model = Model(
    inputs=model.input,
    outputs=[model.get_layer(last_conv_layer_name).output, model.output]
)

# ======================================================
# GRAD-CAM FUNCTION
# ======================================================
def make_gradcam_heatmap(img_array):
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        class_index = tf.argmax(predictions[0])
        loss = predictions[:, class_index]

    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0,1,2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0)
    heatmap /= (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

# ======================================================
# PROCESS IMAGE (with folder structure preservation)
# ======================================================
def process_image(img_path, input_folder, output_folder):
    # Relative path to preserve subfolder structure
    rel_path = os.path.relpath(img_path, input_folder)
    rel_dir = os.path.dirname(rel_path)
    out_dir = os.path.join(output_folder, rel_dir)
    os.makedirs(out_dir, exist_ok=True)

    # Read and preprocess
    img = cv2.imread(img_path)
    img_resized = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img_array = np.expand_dims(img_resized / 255.0, axis=0)

    # Generate Grad-CAM heatmap
    heatmap = make_gradcam_heatmap(img_array)

    # Resize heatmap to original image size
    h, w = img.shape[:2]
    heatmap_resized = cv2.resize(heatmap, (w, h))
    heatmap_uint8 = np.uint8(255 * heatmap_resized)

    # Apply JET colormap and swap red & blue channels
    heatmap_color = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    heatmap_color = heatmap_color[..., ::-1]  # Swap red and blue

    # Overlay heatmap on original image
    overlay = cv2.addWeighted(img, 0.6, heatmap_color, 0.4, 0)

    # Save images
    base_name = os.path.splitext(os.path.basename(img_path))[0]
    cv2.imwrite(os.path.join(out_dir, base_name + "_heatmap.jpg"), heatmap_color)
    cv2.imwrite(os.path.join(out_dir, base_name + "_overlay.jpg"), overlay)

    # Display
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 3, 1)
    plt.title("Original")
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.title("Grad-CAM Heatmap")
    plt.imshow(cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB))
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.title("Overlay")
    plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
    plt.axis("off")

    plt.tight_layout()
    plt.show()

# ======================================================
# RUN ON FOLDER
# ======================================================
for root, dirs, files in os.walk(input_folder):
    for file in files:
        if file.lower().endswith((".jpg", ".png", ".jpeg")):
            process_image(os.path.join(root, file), input_folder, output_folder)

print("✅ Grad-CAM generation completed with swapped colors and folder structure preserved")